What's an Agent?
An agent is an AI system that doesn't just generate text—it thinks, decides, and acts. Instead of answering a question directly, it:

Understands the task ("Find me the current weather and top news stories")
Reasons about what to do ("I need to search the web, then summarize results")
Calls tools (Search the web, fetch articles, parse data)
Observes what happened (Reads the search results)
Decides next steps (Do I have enough info? Or do I need another search?)
Repeats until the task is done

Tool Calling Mechanics :
Here's how tools work from the ground up:
Without tools (limited):
You: "What's the weather?"
Model: "I don't know—my knowledge is from 2025."
With tools (powerful):
You: "What's the weather?"
Model: "I'll check for you. Calling: get_weather(location='London')"
System: [Returns: "London, 22°C, rainy"]
Model: "It's 22°C and rainy in London right now."

Schema Design (Telling Tools What They Do):
A schema is a "instruction manual" that tells the model:

What tools exist,
What each tool does,
What inputs it accepts,
What it returns.

Parallel Calls


Parallel calls mean running multiple tools at the same time instead of one after another.

In [1]:
import os
import boto3 
from dotenv import load_dotenv
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
load_dotenv("myenv.env")
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [2]:
import json

query = "What is the weather in New York today?"

prompt = f"""
Question:
{query}

Answer:
"""

body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.2
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)

response = json.loads(response["body"].read())

print(response["output"]["message"]["content"][0]["text"])

I don't have real-time capabilities to provide current weather information. For up-to-date weather in New York, you can check a reliable weather website or app, such as the Weather Channel, AccuWeather, or a local meteorological service. You can also use a voice assistant like Siri, Google Assistant, or Alexa to get the latest weather forecast for New York.


In [3]:
#TOOL
def get_weather(city):
    # Pretend this is an external API
    weather_data = {
        "New York": "18°C and Cloudy",
        "London": "15°C and Rainy",
        "Delhi": "35°C and Sunny"
    }

    return weather_data.get(city, "Weather not available")

In [4]:
import json

def llm(question):
    # Simulating what an LLM would return
    if "weather" in question.lower():
        return {
            "tool": "get_weather",
            "arguments": {
                "city": "New York"
            }
        }

    return {"answer": "I don't know."}

In [5]:
question = "What is the weather in New York today?"

# Step 1: Ask LLM what to do
llm_response = llm(question)

print("LLM Response:")
print(json.dumps(llm_response, indent=4))

# Step 2: Agent checks if a tool is needed
if llm_response.get("tool") == "get_weather":
    city = llm_response["arguments"]["city"]

    # Step 3: Call the tool
    weather = get_weather(city)

    # Step 4: Give result back to LLM
    final_answer = (
        f"According to the weather service, "
        f"the weather in {city} is {weather}."
    )

    print("\nFinal Answer:")
    print(final_answer)

LLM Response:
{
    "tool": "get_weather",
    "arguments": {
        "city": "New York"
    }
}

Final Answer:
According to the weather service, the weather in New York is 18°C and Cloudy.


In [6]:
#calls the llm 
def call_llm(messages, retries=3):
    for attempt in range(retries):
        try:
            response = bedrock_runtime.invoke_model(
                modelId=MODEL_ID,
                body=json.dumps({
                    "messages": messages,
                    "inferenceConfig": {"maxTokens": 200, "temperature": 0.2}
                }),
                contentType="application/json",
                accept="application/json",
            )

            # --- FIX: Read and decode StreamingBody ---
            body_str = response["body"].read().decode("utf-8")
            output = json.loads(body_str)

            return output["output"]["message"]["content"][0]["text"]

        except Exception as e:
            print(f"LLM error (attempt {attempt+1}): {e}")
            time.sleep(2)

    return "LLM failed after retries."


In [7]:
OPENWEATHER_API_KEY = os.getenv("weatherapi.env")

In [8]:
# --- Weather Tool --- using RAWAPI
import requests, time

# Simple city → coordinates mapping (extend as needed)
CITY_COORDS = {
    "Hyderabad": (17.385, 78.486),
    "Delhi": (28.6139, 77.2090),
    "Mumbai": (19.0760, 72.8777),
    "Chennai": (13.0827, 80.2707),
    "Mancherial":(18.8761, 79.4444)
}

def get_weather(city, retries=3):
    if city not in CITY_COORDS:
        return f"Coordinates for {city} not found."

    lat, lon = CITY_COORDS[city]

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&current=temperature_2m,"
        f"relative_humidity_2m,wind_speed_10m"
    )

    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=5)
            data = response.json()
            if response.status_code == 200 and "current" in data:
                current = data["current"]
                temp = current["temperature_2m"]
                humidity = current["relative_humidity_2m"]
                wind = current["wind_speed_10m"]
                return f"{city}: {temp}°C, Humidity {humidity}%, Wind {wind} km/h"
            else:
                return f"Weather error: {data.get('reason', 'Unknown error')}"
        except Exception as e:
            print(f"Weather error (attempt {attempt+1}): {e}")
            time.sleep(2)

    return "Weather tool failed after retries."


In [9]:
# --- Calculator Tool ---
def calculator(expression):
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Calc error: {e}"

In [ ]:
# --- Agent Loop ---
def agent(query):
    # Step 1: Ask LLM what to do
    decision = call_llm([
        {
            "role": "user",
            "content": [
                {"text": f"User asked: {query}. Should I use weather, calculator, or just answer?"}
            ]
        }
    ])
    print("LLM Decision:", decision)

    # Step 2: Decide tool
    if "weather" in decision.lower():
        # Extract city name safely
        city = query.replace("Weather in", "").strip()
        tool_result = get_weather(city)
        final = call_llm([
            {"role": "user", "content": [{"text": f"Tool result: {tool_result}. Please explain nicely."}]}
        ])
        return final

    elif "calculate" in decision.lower() or any(ch.isdigit() for ch in query):
        expr = query.lower().replace("calculate", "").strip()
        tool_result = calculator(expr)
        final = call_llm([
            {"role": "user", "content": [{"text": f"Tool result: {tool_result}. Please explain nicely."}]}
        ])
        return final

    else:
        # No tool needed
        return decision
# --- Demo ---
#print(agent("What is Python?"))          # Pure LLM
#print(agent("20*20+20"))      # Calculator tool
#print(agent("Weather in Mancherial")) 
#print(agent("who won ipl 2022")) 
print(agent("a person has 10 mangoes he distributed those 10 mangoes 5 persons equally calculate how many mangoes does each person get?")) 
  # Weather tool

LLM Decision: To answer the question, you don't need a calculator or weather information. You simply need to perform a basic division.

Here's the calculation:

The person has 10 mangoes and wants to distribute them equally among 5 persons.

To find out how many mangoes each person gets, you divide the total number of mangoes by the number of persons:

\[ \text{Number of mangoes each person gets} = \frac{10 \text{ mangoes}}{5 \text{ persons}} = 2 \text{ mangoes per person} \]

So, each person gets 2 mangoes.

You don't need a calculator for this simple division, and weather information is irrelevant to this problem. Just the straightforward division is sufficient.
Sure, let's break this down step by step.

You have 10 mangoes that you want to distribute equally among 5 people. To find out how many mangoes each person gets, you need to divide the total number of mangoes by the number of people.

Here's the calculation:

\[ \text{Number of mangoes per person} = \frac{\text{Total number o